# 01 — Setup & Smoke Test

Run every cell top-to-bottom. Each section verifies one piece of the data pipeline.
If something fails, fix it here before running `02_download_sp500_history.ipynb`.

## Vorbereitung (einmalig)

1. **Virtuelle Umgebung anlegen** (im Terminal, im Projekt-Root):
   ```powershell
   python -m venv .venv
   .\.venv\Scripts\Activate.ps1
   pip install -r requirements.txt
   ```
2. **API-Keys eintragen**:
   ```powershell
   Copy-Item .env.example .env
   notepad .env
   ```
   - Finnhub: https://finnhub.io (kostenlos, 60 calls/min)
   - FRED: https://fred.stlouisfed.org/docs/api/api_key.html (kostenlos, sofort)
3. **Jupyter-Kernel registrieren** (damit dieses Notebook die venv nutzt):
   ```powershell
   python -m ipykernel install --user --name=trading-bot --display-name "Python (trading-bot)"
   ```
   Dann oben rechts im Notebook den Kernel `Python (trading-bot)` auswählen.

## 1. Projekt-Root in den Pfad legen

Damit `import config` und `from src.data import ...` aus Notebooks heraus funktionieren.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("On path:", str(PROJECT_ROOT) in sys.path)

## 2. Konfiguration & API-Keys prüfen

In [ ]:
import config

print("DATA_DIR:        ", config.DATA_DIR)
print("PRICES_DIR:      ", config.PRICES_DIR)
print("NEWS_DIR:        ", config.NEWS_DIR)
print("MACRO_DIR:       ", config.MACRO_DIR)
print()
print("FINNHUB_API_KEY: ", "OK" if config.FINNHUB_API_KEY else "NOT SET")
print("FRED_API_KEY:    ", "OK" if config.FRED_API_KEY else "NOT SET")

## 3. S&P 500 Tickerliste von Wikipedia ziehen

In [ ]:
from src.data import universe

sp500 = universe.load_sp500(refresh=True)
print(f"Anzahl Aktien: {len(sp500)}")
print(f"Sektoren:      {sp500['gics_sector'].nunique()}")
sp500.head()

In [ ]:
sp500['gics_sector'].value_counts()

## 4. Smoke-Test: Kursdaten für **eine** Aktie

Bevor wir den Voll-Download (~500 Tickers, ~30 Sekunden) starten,
prüfen wir mit **AAPL** dass yfinance erreichbar ist.

In [ ]:
from src.data import prices

aapl = prices.download_one("AAPL", start="2024-01-01")
print(f"Zeilen:    {len(aapl)}")
print(f"Spalten:   {list(aapl.columns)}")
print(f"Zeitraum:  {aapl.index.min().date()}  ->  {aapl.index.max().date()}")
aapl.tail()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
aapl["adj_close"].plot(ax=ax, title="AAPL — Adjusted Close")
ax.set_ylabel("USD")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Smoke-Test: Storage-Roundtrip

In [ ]:
from src.storage import db

path = db.write_prices("AAPL", aapl)
print(f"Geschrieben nach: {path}")

loaded = db.read_prices("AAPL")
print(f"Wieder gelesen:   {len(loaded)} Zeilen")
print(f"Identisch:        {aapl.equals(loaded)}")

## 6. Smoke-Test: News (Finnhub)

Überspringt sich, falls kein Finnhub-Key gesetzt ist.

In [ ]:
if config.FINNHUB_API_KEY:
    from src.data import news
    aapl_news = news.fetch_company_news("AAPL", days_back=7)
    print(f"News (letzte 7 Tage): {len(aapl_news)}")
    display(aapl_news[["datetime", "source", "headline"]].head(10))
else:
    print("Skip — FINNHUB_API_KEY nicht gesetzt.")

## 7. Smoke-Test: Makrodaten (FRED)

Holt nur den VIX als Single-Series-Test.

In [ ]:
if config.FRED_API_KEY:
    from src.data import macro
    vix = macro.fetch_series("VIXCLS", start="2019-01-01")
    print(f"VIX-Datenpunkte: {len(vix)}")
    print(f"Zeitraum:        {vix.index.min().date()}  ->  {vix.index.max().date()}")

    fig, ax = plt.subplots(figsize=(10, 4))
    vix.plot(ax=ax, title="VIX seit 2019 — Sieh den COVID-Spike März 2020")
    ax.set_ylabel("VIX")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Skip — FRED_API_KEY nicht gesetzt.")

## Wenn alles grün ist

Weiter mit **`02_download_sp500_history.ipynb`** — das holt Kurse + Makrodaten
für den gesamten S&P 500 über die letzten 15 Jahre und legt sie als Parquet ab.

Erwartete Größe: ~150 MB an Kursdaten, läuft in 1–2 Minuten durch.